## Section 1: Setup

In [ ]:
# Cell 1.1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 1.2 — Clone repo and install
# For private repo: add GITHUB_TOKEN in Colab Secrets (🔑 icon in left sidebar)
import os
from google.colab import userdata

try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None

if not os.path.exists("cfd9"):
    if token:
        !git clone https://{token}@github.com/Sovenski/cfd9.git
    else:
        !git clone https://github.com/Sovenski/cfd9.git
%cd cfd9
!pip install -q -r requirements.txt

In [ ]:
# Cell 1.3 — Imports
import numpy as np
import pandas as pd
import optuna
from optuna.samplers import TPESampler, CmaEsSampler
from optuna.pruners import MedianPruner
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from tqdm.auto import tqdm
import json
from pathlib import Path

from src import (
    Params, SpeculatorDetector, load_data, load_cross_asset,
    temporal_split, walk_forward_folds, build_optuna_objective,
    add_pivot_labels, compute_side_score, FOLD_DEFINITIONS,
)

## Section 2: Data

In [ ]:
# Cell 2.1 — Load SPX 1D
SPX_PATH = "data/raw/SPX_1D_18710201_20260318.csv"
df_spx = load_data(SPX_PATH)
print(f"SPX 1D: {len(df_spx)} bars, {df_spx.index[0].date()} – {df_spx.index[-1].date()}")
df_spx.tail()

In [ ]:
# Cell 2.2 — Validate columns and date coverage
required_cols = ['open', 'high', 'low', 'close', 'volume']
missing = [c for c in required_cols if c not in df_spx.columns]
assert not missing, f"Missing columns: {missing}"

# Show basic stats
df_spx.describe()

## Section 3: Verify (Sanity Check with Gold Preset)

In [ ]:
# Cell 3.1 — Run detector with Gold 1D Current preset (default Params)
params_gold = Params()
print("Running detector with Gold 1D Current preset...")
result_gold = SpeculatorDetector(df_spx.reset_index(drop=True), params_gold).run()
n_sh = result_gold['strong_high'].sum()
n_sl = result_gold['strong_low'].sum()
n_rh = (result_gold['signal_high'] & ~result_gold['strong_high']).sum()
n_rl = (result_gold['signal_low'] & ~result_gold['strong_low']).sum()
print(f"Strong HIGH: {n_sh} | Strong LOW: {n_sl}")
print(f"Regular HIGH: {n_rh} | Regular LOW: {n_rl}")

In [ ]:
# Cell 3.2 — Plot: close price with signals and baseline pivots
fig, ax = plt.subplots(figsize=(18, 6))
close = df_spx['close'].values
ax.plot(close, lw=0.8, color='gray', alpha=0.7, label='Close')

idx = np.arange(len(df_spx))
# Baseline pivots
bp_h = result_gold['baseline_ph']
bp_l = result_gold['baseline_pl']
ph_idx = np.where(~np.isnan(bp_h.values))[0]
pl_idx = np.where(~np.isnan(bp_l.values))[0]
ax.scatter(ph_idx, close[ph_idx], marker='v', color='purple', s=15, alpha=0.5, label='Baseline PH')
ax.scatter(pl_idx, close[pl_idx], marker='^', color='purple', s=15, alpha=0.5, label='Baseline PL')

# Signals
sh_idx = np.where(result_gold['strong_high'].values)[0]
sl_idx = np.where(result_gold['strong_low'].values)[0]
rh_idx = np.where(result_gold['signal_high'].values & ~result_gold['strong_high'].values)[0]
rl_idx = np.where(result_gold['signal_low'].values & ~result_gold['strong_low'].values)[0]
ax.scatter(sh_idx, close[sh_idx], marker='v', color='red', s=60, zorder=5, label='Strong HIGH')
ax.scatter(sl_idx, close[sl_idx], marker='^', color='green', s=60, zorder=5, label='Strong LOW')
ax.scatter(rh_idx, close[rh_idx], marker='v', color='orange', s=20, alpha=0.7, label='Regular HIGH')
ax.scatter(rl_idx, close[rl_idx], marker='^', color='teal', s=20, alpha=0.7, label='Regular LOW')
ax.set_title("SPX 1D \u2014 Gold Preset Signals vs Baseline Pivots")
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

## Section 4: Optimize

In [ ]:
# Cell 4.1 — Define parameter space and params_from_trial

import dataclasses

STORAGE = "sqlite:////content/drive/MyDrive/cfd9/optuna.db"
N_TRIALS = 500


def _params_fields(p: Params) -> dict:
    """Extract all fields from a Params dataclass as a dict."""
    return {f.name: getattr(p, f.name) for f in dataclasses.fields(p)}


def params_from_trial(trial: optuna.Trial, side: str) -> Params:
    """Sample Params from an Optuna trial for one study side."""
    if side not in ("high", "low"):
        raise ValueError(f"side must be 'high' or 'low', got {side!r}")
    s = side  # "high" or "low"

    # Integer params
    S_detect = trial.suggest_int(f"{s}_S_detect", 5, 60)
    scale_start = trial.suggest_int(f"{s}_scale_start", 2, 30)
    scale_end = trial.suggest_int(f"{s}_scale_end", 100, 500)
    scale_step = trial.suggest_int(f"{s}_scale_step", 2, 20)
    min_duration = trial.suggest_int(f"{s}_min_duration", 1, 20)
    cooldown_bars = trial.suggest_int(f"{s}_cooldown_bars", 1, 20)
    price_gate_lb = trial.suggest_int(f"{s}_price_gate_lb", 5, 100)
    vola_range_len = trial.suggest_int(f"{s}_vola_range_len", 20, 200)
    er_period = trial.suggest_int(f"{s}_er_period", 5, 60)
    confirm_count = trial.suggest_int(f"{s}_confirm_count", 1, 5)
    pivot_drift_lb = trial.suggest_int(f"{s}_pivot_drift_lb", 2, 20)
    pivot_drift_confirm_bias = trial.suggest_int(f"{s}_pivot_drift_confirm_bias", 0, 2)

    # Float params
    pct_extreme = trial.suggest_float(f"{s}_pct_extreme", 0.70, 0.99)
    min_agreement = trial.suggest_float(f"{s}_min_agreement", 0.10, 0.90)
    dur_extreme_pct = trial.suggest_float(f"{s}_dur_extreme_pct", 0.50, 0.99)
    vol_surge_thresh = trial.suggest_float(f"{s}_vol_surge_thresh", 1.0, 3.0)
    scale_div_thresh = trial.suggest_float(f"{s}_scale_div_thresh", 0.10, 0.60)
    slope_thresh = trial.suggest_float(f"{s}_slope_thresh", 0.01, 0.50)
    vola_high_pct = trial.suggest_float(f"{s}_vola_high_pct", 0.50, 0.99)
    pivot_drift_thresh = trial.suggest_float(f"{s}_pivot_drift_thresh", 0.001, 0.050)
    pivot_drift_gate_mult = trial.suggest_float(f"{s}_pivot_drift_gate_mult", 1.0, 10.0)
    momentum_velocity_thresh = trial.suggest_float(f"{s}_momentum_velocity_thresh", 0.0, 0.05)
    gjr_vote_thresh = trial.suggest_float(f"{s}_gjr_vote_thresh", 0.05, 0.50)
    har_vote_thresh = trial.suggest_float(f"{s}_har_vote_thresh", 0.05, 0.50)

    # Bool params
    er_directional = trial.suggest_categorical(f"{s}_er_directional", [True, False])
    use_trend = trial.suggest_categorical(f"{s}_use_trend", [True, False])
    use_volume = trial.suggest_categorical(f"{s}_use_volume", [True, False])
    use_momentum = trial.suggest_categorical(f"{s}_use_momentum", [True, False])
    use_momentum_velocity = trial.suggest_categorical(f"{s}_use_momentum_velocity", [True, False])
    use_volatility = trial.suggest_categorical(f"{s}_use_volatility", [True, False])
    use_er_gate = trial.suggest_categorical(f"{s}_use_er_gate", [True, False])
    use_gjr_asym = trial.suggest_categorical(f"{s}_use_gjr_asym", [True, False])
    use_har_vol = trial.suggest_categorical(f"{s}_use_har_vol", [True, False])

    # Categorical params
    vola_method = trial.suggest_categorical(f"{s}_vola_method", ["ATR", "StdDev", "Intraday"])
    momentum_velocity_mode = trial.suggest_categorical(f"{s}_momentum_velocity_mode", ["Trend", "Reversal"])

    # Build override dict for the relevant side
    kwargs_high = dict(
        S_detect_high=S_detect, scale_start_high=scale_start, scale_end_high=scale_end,
        scale_step_high=scale_step, min_duration_high=min_duration, cooldown_bars_high=cooldown_bars,
        price_gate_lb_high=price_gate_lb, vola_range_len_high=vola_range_len, er_period_high=er_period,
        pct_extreme_high=pct_extreme, min_agreement_high=min_agreement, dur_extreme_pct_high=dur_extreme_pct,
        confirm_count_high=confirm_count, vol_surge_thresh_high=vol_surge_thresh,
        scale_div_thresh_high=scale_div_thresh, slope_thresh_high=slope_thresh,
        vola_high_pct_high=vola_high_pct, pivot_drift_lookback_high=pivot_drift_lb,
        pivot_drift_thresh_high=pivot_drift_thresh, pivot_drift_gate_mult_high=pivot_drift_gate_mult,
        pivot_drift_confirm_bias_high=pivot_drift_confirm_bias,
        momentum_velocity_thresh_high=momentum_velocity_thresh,
        er_directional_high=er_directional, use_trend_high=use_trend, use_volume_high=use_volume,
        use_momentum_high=use_momentum, use_momentum_velocity_high=use_momentum_velocity,
        use_volatility_high=use_volatility, use_er_gate_high=use_er_gate,
        use_gjr_asym_high=use_gjr_asym, use_har_vol_high=use_har_vol,
        gjr_vote_thresh_high=gjr_vote_thresh, har_vote_thresh_high=har_vote_thresh,
        vola_method_high=vola_method, momentum_velocity_mode_high=momentum_velocity_mode,
    )
    kwargs_low = dict(
        S_detect_low=S_detect, scale_start_low=scale_start, scale_end_low=scale_end,
        scale_step_low=scale_step, min_duration_low=min_duration, cooldown_bars_low=cooldown_bars,
        price_gate_lb_low=price_gate_lb, vola_range_len_low=vola_range_len, er_period_low=er_period,
        pct_extreme_low=pct_extreme, min_agreement_low=min_agreement, dur_extreme_pct_low=dur_extreme_pct,
        confirm_count_low=confirm_count, vol_surge_thresh_low=vol_surge_thresh,
        scale_div_thresh_low=scale_div_thresh, slope_thresh_low=slope_thresh,
        vola_high_pct_low=vola_high_pct, pivot_drift_lookback_low=pivot_drift_lb,
        pivot_drift_thresh_low=pivot_drift_thresh, pivot_drift_gate_mult_low=pivot_drift_gate_mult,
        pivot_drift_confirm_bias_low=pivot_drift_confirm_bias,
        momentum_velocity_thresh_low=momentum_velocity_thresh,
        er_directional_low=er_directional, use_trend_low=use_trend, use_volume_low=use_volume,
        use_momentum_low=use_momentum, use_momentum_velocity_low=use_momentum_velocity,
        use_volatility_low=use_volatility, use_er_gate_low=use_er_gate,
        use_gjr_asym_low=use_gjr_asym, use_har_vol_low=use_har_vol,
        gjr_vote_thresh_low=gjr_vote_thresh, har_vote_thresh_low=har_vote_thresh,
        vola_method_low=vola_method, momentum_velocity_mode_low=momentum_velocity_mode,
    )

    base_dict = _params_fields(Params())
    overrides = kwargs_high if side == "high" else kwargs_low
    return Params(**{**base_dict, **overrides})

In [ ]:
# Cell 4.2 — CMA-ES transition callback

def make_sampler_switch_callback(study_name: str, storage: str):
    """Callback that switches from TPE to CMA-ES after 50 trials."""
    def callback(study: optuna.Study, trial: optuna.FrozenTrial) -> None:
        n_complete = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
        if n_complete >= 50 and not isinstance(study.sampler, CmaEsSampler):
            best = study.best_trial
            # Continuous/integer params only (exclude bool/categorical)
            bool_cat_keys = {k for k, v in best.params.items()
                           if isinstance(v, bool) or isinstance(v, str)}
            x0 = {k: v for k, v in best.params.items() if k not in bool_cat_keys}

            print(f"Switching to CMA-ES after trial {trial.number}. Best score so far: {best.value:.4f}")
            study.sampler = CmaEsSampler(
                x0=x0,
                warn_independent_sampling=False,
            )
    return callback

In [ ]:
# Cell 4.3 — Run optimization

def run_study(side: str, df: pd.DataFrame, n_trials: int = N_TRIALS) -> optuna.Study:
    """Create or resume an Optuna study and run optimization."""
    study_name = f"speculatores_{side}"

    study = optuna.create_study(
        study_name=study_name,
        storage=STORAGE,
        direction="maximize",
        sampler=TPESampler(n_startup_trials=50, seed=42),
        pruner=MedianPruner(n_startup_trials=20, n_warmup_steps=2),
        load_if_exists=True,
    )
    n_existing = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
    if n_existing:
        print(f"Resuming '{study_name}' with {n_existing} complete trials")
    else:
        print(f"Created new study '{study_name}'")

    objective = build_optuna_objective(df, params_from_trial, side)
    switch_cb = make_sampler_switch_callback(study_name, STORAGE)

    n_remaining = max(0, n_trials - len(
        [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    ))
    print(f"Running {n_remaining} more trials for '{side}'...")

    study.optimize(
        objective,
        n_trials=n_remaining,
        callbacks=[switch_cb],
        show_progress_bar=True,
    )
    return study


# Run both studies
study_high = run_study("high", df_spx)
study_low = run_study("low", df_spx)

## Section 5: Results

In [ ]:
# Cell 5.1 — Best params tables

def show_best_params(study: optuna.Study, side: str):
    """Display best trial params for a study side."""
    best = study.best_trial
    print(f"\n{'='*60}")
    print(f"Best trial for {side.upper()}: #{best.number}, score = {best.value:.4f}")
    print(f"{'='*60}")
    df_params = pd.DataFrame.from_dict(best.params, orient='index', columns=['value'])
    df_params.index = df_params.index.str.replace(f"^{side}_", "", regex=True)
    return df_params


df_best_high = show_best_params(study_high, "high")
df_best_low = show_best_params(study_low, "low")
display(df_best_high)
display(df_best_low)

In [ ]:
# Cell 5.2 — Walk-forward fold breakdown

def evaluate_folds(study: optuna.Study, side: str, df: pd.DataFrame):
    """Evaluate best params on each walk-forward fold."""
    best_params = params_from_trial(study.best_trial, side)
    folds = walk_forward_folds(df)

    rows = []
    for i, (df_is, df_oos) in enumerate(folds):
        df_is_r = df_is.reset_index(drop=True)
        df_oos_r = df_oos.reset_index(drop=True)
        det_is = SpeculatorDetector(df_is_r, best_params).run()
        det_oos = SpeculatorDetector(df_oos_r, best_params).run()
        sig_key = f'signal_{side}'
        is_score = compute_side_score(df_is_r, det_is[sig_key], side)
        oos_score = compute_side_score(df_oos_r, det_oos[sig_key], side)
        rows.append({
            'fold': i + 1,
            'oos_period': f"{FOLD_DEFINITIONS[i][2][:7]} \u2013 {FOLD_DEFINITIONS[i][3][:7]}",
            'is_bars': len(df_is_r),
            'oos_bars': len(df_oos_r),
            'is_signals': int(det_is[sig_key].sum()),
            'oos_signals': int(det_oos[sig_key].sum()),
            'is_score': round(is_score, 4),
            'oos_score': round(oos_score, 4),
            'gap': round(is_score - oos_score, 4),
        })
    return pd.DataFrame(rows)


print("HIGH side fold breakdown:")
display(evaluate_folds(study_high, "high", df_spx))
print("\nLOW side fold breakdown:")
display(evaluate_folds(study_low, "low", df_spx))

In [ ]:
# Cell 5.3 — Optimization history plot

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, study, title in [(axes[0], study_high, "HIGH"), (axes[1], study_low, "LOW")]:
    values = [t.value for t in study.trials if t.value is not None]
    best_so_far = np.maximum.accumulate(values)
    ax.plot(values, alpha=0.4, lw=0.8, color='steelblue', label='Trial score')
    ax.plot(best_so_far, lw=2, color='red', label='Best so far')
    ax.set_title(f"{title} \u2014 Optimization History")
    ax.set_xlabel("Trial")
    ax.set_ylabel("Score")
    ax.legend()
plt.tight_layout()
plt.show()

## Section 6: Cross-Asset

In [ ]:
# Cell 6.1 — Define available instruments

INSTRUMENTS = {
    "DAX_1D":       {"path": "data/raw/DAX_1D_19700102_20260324.csv",    "resample": False},
    "NDX_1M\u21921D":  {"path": "data/raw/NDX_1M_20250203_20260226.csv",    "resample": True},
    "GC1_1M\u21921D":  {"path": "data/raw/GC1_1M_20260215_20260312.csv",    "resample": True},
    "SI1_1M\u21921D":  {"path": "data/raw/SI1_1M_20260215_20260312.csv",    "resample": True},
    "EURUSD_1M\u21921D": {"path": "data/raw/EURUSD_1M_20260215_20260312.csv", "resample": True},
    "WTI_1M\u21921D":  {"path": "data/raw/WTI_1M_20260215_20260312.csv",    "resample": True},
}

In [ ]:
# Cell 6.2 — Score best params on each instrument

def score_instrument(name: str, path: str, resample: bool,
                     params_h: Params, params_l: Params) -> dict:
    """Run detector on a cross-asset instrument and return scores."""
    df = load_cross_asset(path, resample_to_1d=resample)
    df_r = df.reset_index(drop=True)
    det = SpeculatorDetector(df_r, params_h).run()
    high_score = compute_side_score(df_r, det['signal_high'], "high")
    det_l = SpeculatorDetector(df_r, params_l).run()
    low_score = compute_side_score(df_r, det_l['signal_low'], "low")
    note = "resampled 1M\u21921D" if resample else "native 1D"
    return {"instrument": name, "high_score": round(high_score, 4),
            "low_score": round(low_score, 4), "bars": len(df_r), "note": note}


best_params_h = params_from_trial(study_high.best_trial, "high")
best_params_l = params_from_trial(study_low.best_trial, "low")

rows = []
for name, info in INSTRUMENTS.items():
    try:
        row = score_instrument(name, info["path"], info["resample"], best_params_h, best_params_l)
        rows.append(row)
        print(f"\u2713 {name}: HIGH={row['high_score']}, LOW={row['low_score']} ({row['bars']} bars, {row['note']})")
    except Exception as e:
        print(f"\u2717 {name}: {e}")

df_cross = pd.DataFrame(rows)
display(df_cross)

## Section 7: Export

In [ ]:
# Cell 7.1 — Save best Params as JSON

import dataclasses


def params_to_dict(p: Params) -> dict:
    """Convert a Params dataclass to a plain dict."""
    return {f.name: getattr(p, f.name) for f in dataclasses.fields(p)}


output_dir = Path("/content/drive/MyDrive/cfd9/results")
output_dir.mkdir(parents=True, exist_ok=True)

best_dict_h = params_to_dict(best_params_h)
best_dict_l = params_to_dict(best_params_l)

with open(output_dir / "best_params_high.json", "w") as f:
    json.dump(best_dict_h, f, indent=2)
with open(output_dir / "best_params_low.json", "w") as f:
    json.dump(best_dict_l, f, indent=2)

print("Saved best_params_high.json and best_params_low.json to Google Drive")

In [ ]:
# Cell 7.2 — Generate Pine-ready comment block

# Pine parameter names from speculatores_v12_presets_gold.pine
# HIGH side Pine variables (is_gold_next branch)
PINE_HIGH_PARAMS = [
    ("S_detect_high", "S_detect_high"),
    ("scale_start_high", "scale_start_high"),
    ("scale_end_high", "scale_end_high"),
    ("scale_step_high", "scale_step_high"),
    ("min_duration_high", "min_duration_high"),
    ("cooldown_bars_high", "cooldown_bars_high"),
    ("price_gate_lb_high", "price_gate_lb_high"),
    ("vola_range_len_high", "vola_range_len_high"),
    ("er_period_high", "er_period_high"),
    ("pct_extreme_high", "pct_extreme_high"),
    ("min_agreement_high", "min_agreement_high"),
    ("dur_extreme_pct_high", "dur_extreme_pct_high"),
    ("confirm_count_high", "confirm_count_high"),
    ("vol_surge_thresh_high", "vol_surge_thresh_high"),
    ("scale_div_thresh_high", "scale_div_thresh_high"),
    ("slope_thresh_high", "slope_thresh_high"),
    ("vola_high_pct_high", "vola_high_pct_high"),
    ("pivot_drift_lookback_high", "pivot_drift_lookback_high"),
    ("pivot_drift_thresh_high", "pivot_drift_thresh_high"),
    ("pivot_drift_gate_mult_high", "pivot_drift_gate_mult_high"),
    ("pivot_drift_confirm_bias_high", "pivot_drift_confirm_bias_high"),
    ("momentum_velocity_thresh_high", "momentum_velocity_thresh_high"),
    ("er_directional_high", "er_directional_high"),
    ("use_trend_high", "use_trend_high"),
    ("use_volume_high", "use_volume_high"),
    ("use_momentum_high", "use_momentum_high"),
    ("use_momentum_velocity_high", "use_momentum_velocity_high"),
    ("use_volatility_high", "use_volatility_high"),
    ("use_er_gate_high", "use_er_gate_high"),
    ("momentum_velocity_mode_high", "momentum_velocity_mode_high"),
    ("vola_method_high", "vola_method_high"),
]

PINE_LOW_PARAMS = [
    ("S_detect_low", "S_detect_low"),
    ("scale_start_low", "scale_start_low"),
    ("scale_end_low", "scale_end_low"),
    ("scale_step_low", "scale_step_low"),
    ("min_duration_low", "min_duration_low"),
    ("cooldown_bars_low", "cooldown_bars_low"),
    ("price_gate_lb_low", "price_gate_lb_low"),
    ("vola_range_len_low", "vola_range_len_low"),
    ("er_period_low", "er_period_low"),
    ("pct_extreme_low", "pct_extreme_low"),
    ("min_agreement_low", "min_agreement_low"),
    ("dur_extreme_pct_low", "dur_extreme_pct_low"),
    ("confirm_count_low", "confirm_count_low"),
    ("vol_surge_thresh_low", "vol_surge_thresh_low"),
    ("scale_div_thresh_low", "scale_div_thresh_low"),
    ("slope_thresh_low", "slope_thresh_low"),
    ("vola_high_pct_low", "vola_high_pct_low"),
    ("pivot_drift_lookback_low", "pivot_drift_lookback_low"),
    ("pivot_drift_thresh_low", "pivot_drift_thresh_low"),
    ("pivot_drift_gate_mult_low", "pivot_drift_gate_mult_low"),
    ("pivot_drift_confirm_bias_low", "pivot_drift_confirm_bias_low"),
    ("momentum_velocity_thresh_low", "momentum_velocity_thresh_low"),
    ("er_directional_low", "er_directional_low"),
    ("use_trend_low", "use_trend_low"),
    ("use_volume_low", "use_volume_low"),
    ("use_momentum_low", "use_momentum_low"),
    ("use_momentum_velocity_low", "use_momentum_velocity_low"),
    ("use_volatility_low", "use_volatility_low"),
    ("use_er_gate_low", "use_er_gate_low"),
    ("momentum_velocity_mode_low", "momentum_velocity_mode_low"),
    ("vola_method_low", "vola_method_low"),
]

# GJR/HAR params NOT in Pine yet
GJR_HAR_PARAMS = {
    "use_gjr_asym_high", "gjr_vote_thresh_high",
    "use_har_vol_high", "har_vote_thresh_high",
    "use_gjr_asym_low", "gjr_vote_thresh_low",
    "use_har_vol_low", "har_vote_thresh_low",
}


def format_pine_value(v) -> str:
    """Format a Python value for Pine Script output."""
    if isinstance(v, bool):
        return str(v).lower()
    if isinstance(v, float):
        return f"{v:.4f}"
    if isinstance(v, str):
        return f'"{v}"'
    return str(v)


lines = [
    "// ============================================================",
    "// Optimizer output \u2014 paste into is_gold_next branch",
    f"// HIGH: trial {study_high.best_trial.number}, score {study_high.best_trial.value:.6f}",
    f"// LOW:  trial {study_low.best_trial.number}, score {study_low.best_trial.value:.6f}",
    "// ============================================================",
    "",
    "// HIGH side",
]
for py_name, pine_name in PINE_HIGH_PARAMS:
    v = best_dict_h.get(py_name, "???")
    lines.append(f"{pine_name} = {format_pine_value(v)}")

lines += ["", "// LOW side"]
for py_name, pine_name in PINE_LOW_PARAMS:
    v = best_dict_l.get(py_name, "???")
    lines.append(f"{pine_name} = {format_pine_value(v)}")

# Check for GJR/HAR usage
gjr_har_active = any(
    best_dict_h.get(p, False) or best_dict_l.get(p, False)
    for p in ["use_gjr_asym_high", "use_gjr_asym_low", "use_har_vol_high", "use_har_vol_low"]
)

# Always add GJR/HAR block (commented out — not yet in Pine)
lines += [
    "",
    "// ── GJR/HAR parameters (NOT yet in speculatores_v12_presets_gold.pine) ──",
    "// To activate: add GJR/HAR vote computation and update max_votes in Pine.",
]
gjr_har_fields = [
    ("use_gjr_asym_high", best_dict_h),
    ("gjr_vote_thresh_high", best_dict_h),
    ("use_har_vol_high", best_dict_h),
    ("har_vote_thresh_high", best_dict_h),
    ("use_gjr_asym_low", best_dict_l),
    ("gjr_vote_thresh_low", best_dict_l),
    ("use_har_vol_low", best_dict_l),
    ("har_vote_thresh_low", best_dict_l),
]
for field, d in gjr_har_fields:
    v = d.get(field, "???")
    lines.append(f"// {field} = {format_pine_value(v)}")

if gjr_har_active:
    lines.append("// ⚠️  At least one GJR/HAR flag is active — Pine code additions required.")

pine_block = "\n".join(lines)
print(pine_block)

# Save to Drive
with open(output_dir / "pine_preset_block.txt", "w") as f:
    f.write(pine_block)
print("\nSaved pine_preset_block.txt to Google Drive")